# DDRI Representative Station-Hour Evidence Charts

대표 대여소 15개 `station-hour` 실험에서 현재 우세 모델인 `LightGBM_RMSE`를 기준으로 근거 차트와 요약표를 생성한다.

- Train: `2023`
- Validation: `2024`
- Final Train: `2023 + 2024`
- Test: `2025`

이번 노트북의 목적은 모델을 하나 더 찾는 것이 아니라, 현재 결과를 설명할 수 있는 근거 자료를 누적하는 것이다.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')

In [2]:
ROOT = Path('/Users/cheng80/Desktop/ddri_work')
RAW_DATA_DIR = ROOT / 'works/05_prediction_long/data'
OUTPUT_DATA_DIR = ROOT / 'works/05_prediction_long/output/data'
IMAGE_DIR = ROOT / 'works/05_prediction_long/output/images'
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = RAW_DATA_DIR / 'ddri_prediction_long_train_2023_2024.csv'
TEST_PATH = RAW_DATA_DIR / 'ddri_prediction_long_test_2025.csv'
METRICS_PATH = OUTPUT_DATA_DIR / 'ddri_station_hour_model_metrics.csv'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
metrics_df = pd.read_csv(METRICS_PATH)

for df in [train, test]:
    df['date'] = pd.to_datetime(df['date'])
    df['datetime'] = pd.to_datetime(
        df['date'].dt.strftime('%Y-%m-%d') + ' ' + df['hour'].astype(str).str.zfill(2) + ':00:00'
    )

train.shape, test.shape

((263160, 16), (131400, 16))

## 1. LightGBM 기준 예측값 생성

근거 차트는 현재 우세 모델인 `LightGBM_RMSE` 예측값을 기준으로 만든다. 이때 시계열 피처는 `station_id`별로 계산해 경계 누수를 막는다.

In [3]:
combined = pd.concat([train.assign(dataset='train'), test.assign(dataset='test')], ignore_index=True)
combined = combined.sort_values(['station_id', 'datetime']).reset_index(drop=True)

grouped_target = combined.groupby('station_id')['rental_count']
combined['lag_1h'] = grouped_target.shift(1).astype('float32')
combined['lag_24h'] = grouped_target.shift(24).astype('float32')
combined['lag_168h'] = grouped_target.shift(168).astype('float32')
combined['rolling_mean_24h'] = grouped_target.transform(lambda s: s.shift(1).rolling(24).mean()).astype('float32')
combined['rolling_mean_168h'] = grouped_target.transform(lambda s: s.shift(1).rolling(168).mean()).astype('float32')
combined['rolling_std_24h'] = grouped_target.transform(lambda s: s.shift(1).rolling(24).std()).astype('float32')

train_model = combined[combined['dataset'] == 'train'].copy()
test_model = combined[combined['dataset'] == 'test'].copy()

FEATURE_COLUMNS = [
    'station_id', 'station_group', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h', 'rolling_mean_168h', 'rolling_std_24h'
]
CATEGORICAL_COLUMNS = ['station_id', 'station_group', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday']

train_2023 = train_model[train_model['date'] < '2024-01-01'].copy()
valid_2024 = train_model[train_model['date'] >= '2024-01-01'].copy()
full_train = train_model.copy()

X_train = train_2023[FEATURE_COLUMNS].copy()
y_train = train_2023['rental_count'].copy()
X_valid = valid_2024[FEATURE_COLUMNS].copy()
y_valid = valid_2024['rental_count'].copy()
X_full = full_train[FEATURE_COLUMNS].copy()
y_full = full_train['rental_count'].copy()
X_test = test_model[FEATURE_COLUMNS].copy()
y_test = test_model['rental_count'].copy()

for frame in [X_train, X_valid, X_full, X_test]:
    for column in CATEGORICAL_COLUMNS:
        frame[column] = frame[column].astype('category')

model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='regression',
)

model.fit(X_full, y_full, categorical_feature=CATEGORICAL_COLUMNS)
test_pred = model.predict(X_test)

test_eval = test_model[['station_id', 'station_name', 'station_group', 'cluster', 'date', 'hour', 'rental_count']].copy()
test_eval['prediction'] = test_pred
test_eval['residual'] = test_eval['rental_count'] - test_eval['prediction']
test_eval['abs_error'] = test_eval['residual'].abs()
test_eval['squared_error'] = test_eval['residual'] ** 2

{
    'rmse': round(float(mean_squared_error(y_test, test_pred) ** 0.5), 4),
    'mae': round(float(mean_absolute_error(y_test, test_pred)), 4),
    'r2': round(float(r2_score(y_test, test_pred)), 4),
}

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004538 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1527
[LightGBM] [Info] Number of data points in the train set: 263160, number of used features: 18
[LightGBM] [Info] Start training from score 0.724514


{'rmse': 0.8916, 'mae': 0.5454, 'r2': 0.5618}

## 2. 설명용 요약표 생성

시간대, 대표 그룹, 스테이션 단위로 실제값과 예측값 차이를 집계한다.

In [4]:
hourly_summary = test_eval.groupby('hour').agg(
    actual_mean=('rental_count', 'mean'),
    predicted_mean=('prediction', 'mean'),
    mae=('abs_error', 'mean'),
).reset_index()

station_group_summary = test_eval.groupby('station_group').agg(
    actual_mean=('rental_count', 'mean'),
    predicted_mean=('prediction', 'mean'),
    mae=('abs_error', 'mean'),
    rmse=('squared_error', lambda s: float(s.mean() ** 0.5)),
).reset_index().sort_values('mae', ascending=False)

station_summary = test_eval.groupby(['station_id', 'station_name', 'station_group']).agg(
    actual_mean=('rental_count', 'mean'),
    predicted_mean=('prediction', 'mean'),
    mae=('abs_error', 'mean'),
    rmse=('squared_error', lambda s: float(s.mean() ** 0.5)),
).reset_index().sort_values('mae', ascending=False)

residual_summary = test_eval[['residual', 'abs_error']].describe()

hourly_path = OUTPUT_DATA_DIR / 'ddri_station_hour_hourly_actual_vs_predicted.csv'
group_path = OUTPUT_DATA_DIR / 'ddri_station_hour_station_group_error_summary.csv'
station_path = OUTPUT_DATA_DIR / 'ddri_station_hour_station_error_summary.csv'

hourly_summary.to_csv(hourly_path, index=False, encoding='utf-8-sig')
station_group_summary.to_csv(group_path, index=False, encoding='utf-8-sig')
station_summary.to_csv(station_path, index=False, encoding='utf-8-sig')

hourly_summary.head(), station_group_summary, station_summary.head(10)

(   hour  actual_mean  predicted_mean       mae
 0     0     0.194703        0.241890  0.324323
 1     1     0.169132        0.184279  0.277640
 2     2     0.116164        0.133337  0.211603
 3     3     0.090046        0.108765  0.175813
 4     4     0.068858        0.094866  0.152824,
   station_group  actual_mean  predicted_mean       mae      rmse
 1  아침 도착 업무 집중형     1.146233        1.223656  0.755227  1.304946
 2     업무/상업 혼합형     0.521309        0.571766  0.533469  0.800051
 4        주거 도착형     0.555441        0.536580  0.502380  0.801534
 0       생활권 혼합형     0.414155        0.451926  0.485252  0.687250
 3        외곽 주거형     0.458562        0.484964  0.450767  0.719341,
     station_id                station_name station_group  actual_mean  \
 8         2377                    수서역 5번출구  아침 도착 업무 집중형     1.822603   
 5         2348                포스코사거리(기업은행)  아침 도착 업무 집중형     1.475228   
 14        4917                  일원에코파크 주차장        주거 도착형     1.104452   
 7         2359   

## 3. 근거 차트 생성

모델 비교, 시간대 평균 패턴, 그룹별 MAE, residual 분포를 차트로 저장한다.

In [5]:
test_metrics = metrics_df[metrics_df['split'] == 'test_2025_refit'].copy()
plt.figure(figsize=(10, 5))
sns.barplot(data=test_metrics.sort_values('rmse'), x='rmse', y='model', hue='model', legend=False, palette='Blues_r')
plt.title('Representative Station-Hour Model Comparison on 2025 Test')
plt.xlabel('RMSE')
plt.ylabel('Model')
plt.tight_layout()
model_chart_path = IMAGE_DIR / 'ddri_station_hour_model_comparison_test_rmse.png'
plt.savefig(model_chart_path, dpi=150)
plt.close()

plt.figure(figsize=(11, 5))
plt.plot(hourly_summary['hour'], hourly_summary['actual_mean'], marker='o', label='Actual Mean')
plt.plot(hourly_summary['hour'], hourly_summary['predicted_mean'], marker='o', label='Predicted Mean')
plt.title('Representative Station-Hour: Hourly Actual vs Predicted Mean')
plt.xlabel('Hour')
plt.ylabel('Rental Count')
plt.xticks(range(0, 24, 1))
plt.legend()
plt.tight_layout()
hourly_chart_path = IMAGE_DIR / 'ddri_station_hour_hourly_actual_vs_predicted.png'
plt.savefig(hourly_chart_path, dpi=150)
plt.close()

plt.figure(figsize=(10, 5))
sns.barplot(data=station_group_summary, x='mae', y='station_group', hue='station_group', legend=False, palette='crest')
plt.title('Representative Station-Hour: MAE by Station Group')
plt.xlabel('MAE')
plt.ylabel('Station Group')
plt.tight_layout()
group_chart_path = IMAGE_DIR / 'ddri_station_hour_station_group_mae.png'
plt.savefig(group_chart_path, dpi=150)
plt.close()

plt.figure(figsize=(10, 5))
sns.histplot(test_eval['residual'], bins=60, kde=True, color='#2C7FB8')
plt.title('Representative Station-Hour: Residual Distribution on 2025 Test')
plt.xlabel('Residual (Actual - Predicted)')
plt.ylabel('Count')
plt.tight_layout()
residual_chart_path = IMAGE_DIR / 'ddri_station_hour_residual_distribution.png'
plt.savefig(residual_chart_path, dpi=150)
plt.close()

plt.figure(figsize=(6, 6))
sns.scatterplot(data=test_eval.sample(min(len(test_eval), 12000), random_state=42), x='rental_count', y='prediction', alpha=0.25, s=16, color='#1D91C0')
max_value = max(float(test_eval['rental_count'].max()), float(test_eval['prediction'].max()))
plt.plot([0, max_value], [0, max_value], linestyle='--', color='black', linewidth=1)
plt.title('Representative Station-Hour: Actual vs Predicted')
plt.xlabel('Actual Rental Count')
plt.ylabel('Predicted Rental Count')
plt.tight_layout()
scatter_chart_path = IMAGE_DIR / 'ddri_station_hour_actual_vs_predicted_scatter.png'
plt.savefig(scatter_chart_path, dpi=150)
plt.close()

model_chart_path, hourly_chart_path, group_chart_path, residual_chart_path, scatter_chart_path

/var/folders/yk/7gcx2d_d3g36kr1klhhz4q700000gn/T/ipykernel_27973/3095613357.py:30: UserWarning: Glyph 50500 (\N{HANGUL SYLLABLE A}) missing from font(s) Arial.
  plt.tight_layout()
/var/folders/yk/7gcx2d_d3g36kr1klhhz4q700000gn/T/ipykernel_27973/3095613357.py:30: UserWarning: Glyph 52840 (\N{HANGUL SYLLABLE CIM}) missing from font(s) Arial.
  plt.tight_layout()
/var/folders/yk/7gcx2d_d3g36kr1klhhz4q700000gn/T/ipykernel_27973/3095613357.py:30: UserWarning: Glyph 46020 (\N{HANGUL SYLLABLE DO}) missing from font(s) Arial.
  plt.tight_layout()
/var/folders/yk/7gcx2d_d3g36kr1klhhz4q700000gn/T/ipykernel_27973/3095613357.py:30: UserWarning: Glyph 52265 (\N{HANGUL SYLLABLE CAG}) missing from font(s) Arial.
  plt.tight_layout()
/var/folders/yk/7gcx2d_d3g36kr1klhhz4q700000gn/T/ipykernel_27973/3095613357.py:30: UserWarning: Glyph 50629 (\N{HANGUL SYLLABLE EOB}) missing from font(s) Arial.
  plt.tight_layout()
/var/folders/yk/7gcx2d_d3g36kr1klhhz4q700000gn/T/ipykernel_27973/3095613357.py:30: UserW

(PosixPath('/Users/cheng80/Desktop/ddri_work/works/05_prediction_long/output/images/ddri_station_hour_model_comparison_test_rmse.png'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/works/05_prediction_long/output/images/ddri_station_hour_hourly_actual_vs_predicted.png'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/works/05_prediction_long/output/images/ddri_station_hour_station_group_mae.png'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/works/05_prediction_long/output/images/ddri_station_hour_residual_distribution.png'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/works/05_prediction_long/output/images/ddri_station_hour_actual_vs_predicted_scatter.png'))

In [6]:
station_group_summary, station_summary.head(10), residual_summary

(  station_group  actual_mean  predicted_mean       mae      rmse
 1  아침 도착 업무 집중형     1.146233        1.223656  0.755227  1.304946
 2     업무/상업 혼합형     0.521309        0.571766  0.533469  0.800051
 4        주거 도착형     0.555441        0.536580  0.502380  0.801534
 0       생활권 혼합형     0.414155        0.451926  0.485252  0.687250
 3        외곽 주거형     0.458562        0.484964  0.450767  0.719341,
     station_id                station_name station_group  actual_mean  \
 8         2377                    수서역 5번출구  아침 도착 업무 집중형     1.822603   
 5         2348                포스코사거리(기업은행)  아침 도착 업무 집중형     1.475228   
 14        4917                  일원에코파크 주차장        주거 도착형     1.104452   
 7         2359             국립국악중,고교 정문 맞은편        외곽 주거형     0.762100   
 4         2328   르네상스 호텔 사거리 역삼지하보도 7번출구 앞     업무/상업 혼합형     0.649429   
 12        4902                  구역삼세무서 교차로     업무/상업 혼합형     0.682991   
 1         2320                도곡역 대치지구대 방향       생활권 혼합형     0.595205   
 11        